# 長期模擬與視覺化 (Spherical DG Long-Term Simulation)\n
目標：執行 $t_{final} = 12.0$ 的長時間模擬，並透過 Dual-View (2D 攤平視角 與 3D 球面視角) 檢驗 Topological Gluing 與波傳遞的正確性。\n

In [27]:
%matplotlib inline
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.tri as mtri
import time

def _resolve_project_root() -> Path:
    cwd = Path.cwd().resolve()
    if (cwd / "src").exists(): return cwd
    if cwd.name == "experimental" and (cwd.parents[1] / "src").exists(): return cwd.parents[1]
    for parent in cwd.parents:
        if (parent / "src").exists(): return parent
    return cwd

project_root = _resolve_project_root()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

try:
    from src.core.generators import get_reference_data
    from src.core.connectivity import build_connectivity
    from src.bases.vandermonde import vandermonde_2d_dubiner, grad_vandermonde_2d_dubiner
    from src.reconstruction import build_differentiation_matrices, build_fmask_table1
    from src.geometry.metrics import compute_geometric_factors_batch
except ImportError as e:
    print(f"匯入錯誤: {e}. 請確保您的環境中有 src 模組。")
    sys.exit(1)

In [28]:
# ============================================================================
# 階段一與二：網格生成與拓樸縫合 (Topology Gluing)
# ============================================================================

def generate_flattened_8face_mesh(n_div: int):
    C = (0.0, 0.0); TR = (1.0, 1.0); TL = (-1.0, 1.0); BL = (-1.0, -1.0); BR = (1.0, -1.0)
    R = (1.0, 0.0); T = (0.0, 1.0); L = (-1.0, 0.0); B = (0.0, -1.0)
    base_triangles = [(C, R, T), (TR, T, R), (C, T, L), (TL, L, T),
                      (C, L, B), (BL, B, L), (C, B, R), (BR, R, B)]

    nodes, EToV, face_ids = [], [], []
    node_map = {}

    def get_node_id(x, y):
        key = (round(x, 10), round(y, 10))
        if key not in node_map:
            node_map[key] = len(nodes)
            nodes.append([x, y])
        return node_map[key]

    n_segments = 2 ** n_div
    for face_idx, (v0, v1, v2) in enumerate(base_triangles):
        face_id = face_idx + 1
        v0, v1, v2 = np.array(v0), np.array(v1), np.array(v2)

        for i in range(n_segments):
            for j in range(n_segments - i):
                p1 = v0 + (v1-v0)*(i/n_segments) + (v2-v0)*(j/n_segments)
                p2 = v0 + (v1-v0)*((i+1)/n_segments) + (v2-v0)*(j/n_segments)
                p3 = v0 + (v1-v0)*(i/n_segments) + (v2-v0)*((j+1)/n_segments)
                EToV.append([get_node_id(*p1), get_node_id(*p2), get_node_id(*p3)])
                face_ids.append(face_id)
                if i > 0:
                    p4 = v0 + (v1-v0)*(i/n_segments) + (v2-v0)*(j/n_segments)
                    p5 = v0 + (v1-v0)*((i-1)/n_segments) + (v2-v0)*((j+1)/n_segments)
                    p6 = v0 + (v1-v0)*(i/n_segments) + (v2-v0)*((j+1)/n_segments)
                    EToV.append([get_node_id(*p4), get_node_id(*p6), get_node_id(*p5)])
                    face_ids.append(face_id)
    return np.array(nodes), np.array(EToV), np.array(face_ids)

def build_global_index_maps(EToV, EToE, EToF, xi_ref, eta_ref, Np, weights_1d):
    K, nfp = int(EToV.shape[0]), int(len(weights_1d))
    bary_coords = np.column_stack([(-xi_ref - eta_ref)/2.0, (xi_ref + 1.0)/2.0, (eta_ref + 1.0)/2.0])
    fmask = build_fmask_table1(bary_coords)
    vmapM, vmapP = np.zeros((3*nfp, K), dtype=int), np.zeros((3*nfp, K), dtype=int)

    for k_elem in range(K):
        for face in range(3):
            local_nodes = fmask[:, face]
            vmapM[face*nfp:(face+1)*nfp, k_elem] = k_elem * Np + local_nodes
            k_neighbor, f_neighbor = int(EToE[k_elem, face]), int(EToF[k_elem, face])
            if k_neighbor == k_elem:
                vmapP[face*nfp:(face+1)*nfp, k_elem] = k_elem * Np + local_nodes
            else:
                neighbor_local_nodes = fmask[:, f_neighbor]
                vmapP[face*nfp:(face+1)*nfp, k_elem] = (k_neighbor * Np + neighbor_local_nodes)[::-1]
    return vmapM, vmapP, vmapM == vmapP, fmask

def apply_spherical_topology_vmapP(vmapM, vmapP, nodes, EToV, tol=1e-8):
    vmapP_spherical = vmapP.copy()
    Nfp = vmapM.shape[0] // 3
    boundary_faces = []

    for k in range(vmapM.shape[1]):
        for f in range(3):
            idx_start, idx_end = f * Nfp, (f + 1) * Nfp
            if np.all(vmapM[idx_start:idx_end, k] == vmapP[idx_start:idx_end, k]):
                v1, v2 = EToV[k, f], EToV[k, (f+1)%3]
                boundary_faces.append({
                    'k': k, 'f': f, 'mx': (nodes[v1, 0] + nodes[v2, 0]) / 2.0, 'my': (nodes[v1, 1] + nodes[v2, 1]) / 2.0,
                    'nodes': vmapM[idx_start:idx_end, k], 'idx_start': idx_start, 'idx_end': idx_end
                })

    left = [f for f in boundary_faces if np.isclose(f['mx'], -1.0, atol=tol)]
    right = [f for f in boundary_faces if np.isclose(f['mx'], 1.0, atol=tol)]
    bottom = [f for f in boundary_faces if np.isclose(f['my'], -1.0, atol=tol)]
    top = [f for f in boundary_faces if np.isclose(f['my'], 1.0, atol=tol)]

    for faces, is_x in [(right, True), (left, True), (top, False), (bottom, False)]:
        coord_idx = 'my' if is_x else 'mx'
        pos_faces = sorted([f for f in faces if f[coord_idx] > tol], key=lambda f: f[coord_idx])
        neg_faces = sorted([f for f in faces if f[coord_idx] < -tol], key=lambda f: -f[coord_idx])
        for p_face, n_face in zip(pos_faces, neg_faces):
            vmapP_spherical[p_face['idx_start']:p_face['idx_end'], p_face['k']] = n_face['nodes'][::-1]
            vmapP_spherical[n_face['idx_start']:n_face['idx_end'], n_face['k']] = p_face['nodes'][::-1]
    return vmapP_spherical

In [29]:
# ============================================================================
# 階段三：映射與反變速度 (Mapping & Contravariant Velocity)
# ============================================================================

def map_square_to_sphere_exact(nodes, EToV, xi_ref, eta_ref, face_ids, R_sphere):
    K, Np = EToV.shape[0], len(xi_ref)
    lam, theta, X_global, Y_global = np.zeros((Np, K)), np.zeros((Np, K)), np.zeros((Np, K)), np.zeros((Np, K))
    L1, L2, L3 = -(xi_ref + eta_ref)/2.0, (xi_ref + 1.0)/2.0, (eta_ref + 1.0)/2.0

    for k in range(K):
        X = L1 * nodes[EToV[k], 0][0] + L2 * nodes[EToV[k], 0][1] + L3 * nodes[EToV[k], 0][2]
        Y = L1 * nodes[EToV[k], 1][0] + L2 * nodes[EToV[k], 1][1] + L3 * nodes[EToV[k], 1][2]
        X_global[:, k], Y_global[:, k], fid = X, Y, face_ids[k]

        if fid == 1:   x, y, lam_0, theta_sign, is_odd = X, Y, 0.0, 1.0, True
        elif fid == 2: x, y, lam_0, theta_sign, is_odd = 1-X, 1-Y, 0.0, -1.0, False
        elif fid == 3: x, y, lam_0, theta_sign, is_odd = Y, -X, np.pi/2, 1.0, True
        elif fid == 4: x, y, lam_0, theta_sign, is_odd = 1-Y, 1+X, np.pi/2, -1.0, False
        elif fid == 5: x, y, lam_0, theta_sign, is_odd = -X, -Y, np.pi, 1.0, True
        elif fid == 6: x, y, lam_0, theta_sign, is_odd = 1+X, 1+Y, np.pi, -1.0, False
        elif fid == 7: x, y, lam_0, theta_sign, is_odd = -Y, X, 3*np.pi/2, 1.0, True
        elif fid == 8: x, y, lam_0, theta_sign, is_odd = 1+Y, 1-X, 3*np.pi/2, -1.0, False

        r = x + y
        safe_r = np.where(r < 1e-12, 1e-12, r)
        lam[:, k] = lam_0 + (np.pi / 2.0) * ((y if is_odd else x) / safe_r)
        theta[:, k] = theta_sign * np.arcsin(1.0 - np.clip(r**2, 0.0, 1.0))
    return lam, theta, X_global, Y_global

def velocity_solid_body_latlon(lam, theta, u0=1.0, alpha0=0.0):
    return u0*(np.cos(alpha0)*np.cos(theta) + np.sin(alpha0)*np.cos(lam)*np.sin(theta)), -u0*np.sin(alpha0)*np.sin(lam)

def compute_contravariant_velocity_8branch(lam, theta, X, Y, face_ids, R_sphere, u0, alpha0):
    u_phys, v_phys = velocity_solid_body_latlon(lam, theta, u0, alpha0)
    tilde_u, tilde_v = np.zeros_like(u_phys), np.zeros_like(v_phys)
    tan_th = np.tan(np.clip(theta, -np.pi/2 + 1e-12, np.pi/2 - 1e-12))
    u_over_cos = u0 * (np.cos(alpha0) + np.sin(alpha0) * np.cos(lam) * tan_th)
    cos_th = np.cos(theta)

    for fid in range(1, 9):
        mask = (face_ids == fid)[np.newaxis, :]

        if fid == 1:   x, y = X, Y
        elif fid == 2: x, y = 1 - X, 1 - Y
        elif fid == 3: x, y = Y, -X
        elif fid == 4: x, y = 1 - Y, 1 + X
        elif fid == 5: x, y = -X, -Y
        elif fid == 6: x, y = 1 + X, 1 + Y
        elif fid == 7: x, y = -Y, X
        elif fid == 8: x, y = 1 + Y, 1 - X
        else: continue

        r = x + y
        safe_r = np.where(r < 1e-12, 1e-12, r)

        # 修正：第一項改為 (2 * r)，第二項的分母改為 safe_r**2
        if fid % 2 != 0:
            dx = -(2 * r) / (np.pi * R_sphere) * u_over_cos - (x * cos_th) / (2 * R_sphere * safe_r**2) * v_phys
            dy =  (2 * r) / (np.pi * R_sphere) * u_over_cos - (y * cos_th) / (2 * R_sphere * safe_r**2) * v_phys
        else:
            dx =  (2 * r) / (np.pi * R_sphere) * u_over_cos + (x * cos_th) / (2 * R_sphere * safe_r**2) * v_phys
            dy = -(2 * r) / (np.pi * R_sphere) * u_over_cos + (y * cos_th) / (2 * R_sphere * safe_r**2) * v_phys

        if fid == 1:   tu, tv = dx, dy
        elif fid == 2: tu, tv = -dx, -dy
        elif fid == 3: tu, tv = -dy, dx
        elif fid == 4: tu, tv = dy, -dx
        elif fid == 5: tu, tv = -dx, -dy
        elif fid == 6: tu, tv = dx, dy
        elif fid == 7: tu, tv = dy, -dx
        elif fid == 8: tu, tv = -dy, dx

        tilde_u = np.where(mask, tu, tilde_u)
        tilde_v = np.where(mask, tv, tilde_v)
    return tilde_u, tilde_v

In [30]:
# ============================================================================
# 階段四：標準 2D Affine RHS 與 Metrics (Standard 2D Cartesian DG)
# ============================================================================

def compute_metrics_vectorized(nodes, EToV):
    vertices = nodes[EToV]
    metrics = compute_geometric_factors_batch(vertices)
    e01 = np.linalg.norm(vertices[:, 1, :] - vertices[:, 0, :], axis=1)
    e12 = np.linalg.norm(vertices[:, 2, :] - vertices[:, 1, :], axis=1)
    e20 = np.linalg.norm(vertices[:, 0, :] - vertices[:, 2, :], axis=1)
    return metrics["J"], metrics["rx"], metrics["ry"], metrics["sx"], metrics["sy"], np.minimum(np.minimum(e01, e12), e20)

def compute_face_geometry(nodes, EToV, weights_1d):
    K, nfp = int(EToV.shape[0]), int(len(weights_1d))
    J_face, nx_array, ny_array = np.zeros((3, K)), np.zeros((3, K)), np.zeros((3, K))

    for k in range(K):
        v1, v2, v3 = nodes[EToV[k]]
        edges = [v2 - v1, v3 - v2, v1 - v3]
        for face in range(3):
            dx, dy = edges[face]
            length = np.hypot(dx, dy)
            J_face[face, k], nx_array[face, k], ny_array[face, k] = length, dy / length, -dx / length
    return np.repeat(nx_array, nfp, axis=0), np.repeat(ny_array, nfp, axis=0), np.repeat(J_face, nfp, axis=0)

def compute_rhs_vectorized(Q, X, Y, D_r_ref, D_s_ref, E, rx, ry, sx, sy, vmapM, vmapP, weights_2d, J, weights_1d, nx_expanded, ny_expanded, J_face_expanded, M_inv_projected, u_arr, v_arr):
    q = Q[0]
    dq_dr, dq_ds = D_r_ref @ q, D_s_ref @ q
    dq_dx, dq_dy = rx[np.newaxis, :] * dq_dr + sx[np.newaxis, :] * dq_ds, ry[np.newaxis, :] * dq_dr + sy[np.newaxis, :] * dq_ds

    duq_dr, duq_ds = D_r_ref @ (u_arr * q), D_s_ref @ (u_arr * q)
    dvq_dr, dvq_ds = D_r_ref @ (v_arr * q), D_s_ref @ (v_arr * q)
    term1_x = rx[np.newaxis, :] * duq_dr + sx[np.newaxis, :] * duq_ds
    term1_y = ry[np.newaxis, :] * dvq_dr + sy[np.newaxis, :] * dvq_ds
    term2 = u_arr * dq_dx + v_arr * dq_dy
    term3 = (rx[np.newaxis, :] * (D_r_ref @ u_arr) + sx[np.newaxis, :] * (D_s_ref @ u_arr) + ry[np.newaxis, :] * (D_r_ref @ v_arr) + sy[np.newaxis, :] * (D_s_ref @ v_arr)) * q
    volume_term = -0.5 * (term1_x + term1_y + term2 + term3)

    q_flat, u_boundary, v_boundary = q.flatten(order="F"), u_arr.flatten(order="F")[vmapM], v_arr.flatten(order="F")[vmapM]
    q_M, q_P = q_flat[vmapM], q_flat[vmapP]
    v_normal = nx_expanded * u_boundary + ny_expanded * v_boundary
    flux_penalty = 0.5 * (v_normal - np.abs(v_normal)) * (q_M - q_P)

    scaled_penalty = flux_penalty * np.tile(weights_1d, 3)[:, np.newaxis] * J_face_expanded
    surface_term = (0.5 / J[np.newaxis, :]) * (M_inv_projected @ (E.T @ scaled_penalty))
    return (volume_term + surface_term)[np.newaxis, :, :]

def latlon_to_cartesian(lam, theta, R):
    return R * np.cos(theta) * np.cos(lam), R * np.cos(theta) * np.sin(lam), R * np.sin(theta)

def exact_gaussian_bell_latlon(lam, theta, t, u0, R):
    x, y, z = latlon_to_cartesian(lam, theta, R)
    Omega = u0 / R
    xc, yc, zc = -R * np.sin(Omega * t), R * np.cos(Omega * t), 0.0
    dot_product = (x * xc + y * yc + z * zc) / (R**2)
    return np.exp(-10.0 * (R * np.arccos(np.clip(dot_product, -1.0, 1.0)))**2)

In [31]:
# ============================================================================
# 階段五：長期模擬與雙重視角繪圖 (Long-Term Sim & Dual-View Visualization)
# ============================================================================

A_RK = np.array([0.0, -567301805773.0 / 1357537059087.0, -2404267990393.0 / 2016746695238.0,
                 -3550918686646.0 / 2091501179385.0, -1275806237668.0 / 842570457699.0])
B_RK = np.array([1432997174477.0 / 9575080441755.0, 5161836677717.0 / 13612068292357.0,
                 1720146321549.0 / 2090206949498.0, 3134564353537.0 / 4481467310338.0,
                 2277821191437.0 / 14882151754819.0])

def run_longterm_simulation_and_plot(k_degree=4, n_div=4, t_final=12.0, CFL=0.5):
    print("\n" + "=" * 60)
    print(f"Spherical DG Long-Term Advection (n_div={n_div}, k={k_degree}, t_final={t_final})")
    print("=" * 60)

    R_sphere = 1.0
    nodes, EToV, face_ids = generate_flattened_8face_mesh(n_div)
    K = EToV.shape[0]
    print(f"Mesh: {nodes.shape[0]} nodes, {K} elements. Generating matrices...")

    EToE, EToF = build_connectivity(EToV)
    ref_data = get_reference_data("table1", k_degree)
    xi_ref, eta_ref = ref_data["xi"], ref_data["eta"]
    weights_ref, weights_1d = ref_data["weights"], ref_data["weights_1d"]
    Np, nfp = len(xi_ref), len(weights_1d)

    vmapM, vmapP, _, fmask = build_global_index_maps(EToV, EToE, EToF, xi_ref, eta_ref, Np, weights_1d)
    vmapP_periodic = apply_spherical_topology_vmapP(vmapM, vmapP, nodes, EToV)

    V_nodal = vandermonde_2d_dubiner(xi_ref, eta_ref, k_degree)
    Vr, Vs = grad_vandermonde_2d_dubiner(xi_ref, eta_ref, k_degree)
    D_r_ref, D_s_ref = build_differentiation_matrices(V_nodal, Vr, Vs, w=weights_ref)
    M_inv_projected = V_nodal @ np.linalg.inv(V_nodal.T @ np.diag(weights_ref) @ V_nodal) @ V_nodal.T

    E = np.zeros((3 * nfp, Np))
    for face in range(3): E[face * nfp + np.arange(nfp), fmask[:, face]] = 1.0

    J, rx, ry, sx, sy, h_min_array = compute_metrics_vectorized(nodes, EToV)
    nx_expanded, ny_expanded, J_face_expanded = compute_face_geometry(nodes, EToV, weights_1d)
    lam, theta, X_global, Y_global = map_square_to_sphere_exact(nodes, EToV, xi_ref, eta_ref, face_ids, R_sphere)
    u0, alpha0 = 1.0, 0.0
    tilde_u, tilde_v = compute_contravariant_velocity_8branch(lam, theta, X_global, Y_global, face_ids, R_sphere, u0, alpha0)

    dt_global = CFL * float(np.min(h_min_array)) / (float(np.max(np.sqrt(tilde_u**2 + tilde_v**2))) * (k_degree + 1)**2)
    print(f"Global dt = {dt_global:.6e}")

    def rhs_func(Q_state):
        return compute_rhs_vectorized(Q_state, X_global, Y_global, D_r_ref, D_s_ref, E, rx, ry, sx, sy, vmapM, vmapP_periodic, weights_ref, J, weights_1d, nx_expanded, ny_expanded, J_face_expanded, M_inv_projected, tilde_u, tilde_v)

    Q = exact_gaussian_bell_latlon(lam, theta, 0.0, u0, R_sphere)[np.newaxis, :, :]
    M_diag = weights_ref[:, np.newaxis] * J[np.newaxis, :] * (np.pi * R_sphere**2 / 4.0)

    t_now, step_count, du = 0.0, 0, np.zeros_like(Q)
    start_time = time.time()
    print(f"Starting Time Integration (approx {int(t_final/dt_global)} steps)...")

    while t_now < t_final - 1e-12:
        current_dt = min(dt_global, t_final - t_now)
        du.fill(0.0)
        for stage in range(5):
            du = A_RK[stage] * du + current_dt * rhs_func(Q)
            Q = Q + B_RK[stage] * du
        t_now += current_dt
        step_count += 1

        if step_count % 500 == 0 or t_now >= t_final - 1e-12:
            err = float(np.sqrt(np.sum((Q[0] - exact_gaussian_bell_latlon(lam, theta, t_now, u0, R_sphere))**2 * M_diag)))
            print(f"  Step {step_count:5d} | Time {t_now:6.3f} | L2 Error: {err:.4e} | Elapsed: {time.time()-start_time:.1f}s")

    print(f"Simulation Finished! Plotting results...")

    # Build exact triangulation to avoid Delaunay crashes
    local_tri = mtri.Triangulation(xi_ref, eta_ref).triangles
    global_tri = np.vstack([local_tri + k * Np for k in range(K)])

    X_flat, Y_flat = X_global.flatten(order='F'), Y_global.flatten(order='F')
    Q_flat = Q[0].flatten(order='F')
    lam_flat, theta_flat = lam.flatten(order='F'), theta.flatten(order='F')

    triang = mtri.Triangulation(X_flat, Y_flat, triangles=global_tri)

    fig = plt.figure(figsize=(16, 7))

    # 2D Flattened View
    ax1 = fig.add_subplot(121)
    tc1 = ax1.tricontourf(triang, Q_flat, levels=50, cmap='viridis')
    fig.colorbar(tc1, ax=ax1, shrink=0.7)
    ax1.set_title(f"2D Flattened Square View (t={t_final})", fontsize=14)
    ax1.set_xlabel("X")
    ax1.set_ylabel("Y")
    ax1.set_aspect('equal')

    # 3D Spherical View
    # x_3d, y_3d, z_3d = latlon_to_cartesian(lam_flat, theta_flat, R_sphere)
    # ax2 = fig.add_subplot(122, projection='3d')
    # surf = ax2.plot_trisurf(x_3d, y_3d, z_3d, triangles=global_tri, cmap='viridis', antialiased=True)
    # surf.set_array(Q_flat)  # Map colors to Q instead of Z
    # surf.autoscale()
    # fig.colorbar(surf, ax=ax2, shrink=0.7)
    # ax2.set_title(f"3D Spherical View (t={t_final})", fontsize=14)
    # ax2.view_init(elev=20, azim=45)

    plt.tight_layout()
    plt.savefig('longterm_simulation_result.png', dpi=150)
    plt.close(fig) # ensure the plot doesn't hang in terminal

if __name__ == "__main__":
    run_longterm_simulation_and_plot(k_degree=4, n_div=4, t_final=3.14, CFL=1)


Spherical DG Long-Term Advection (n_div=4, k=4, t_final=3.14)
Mesh: 1089 nodes, 2048 elements. Generating matrices...
Global dt = 2.776802e-03
Starting Time Integration (approx 1130 steps)...
  Step   500 | Time  1.388 | L2 Error: 9.9471e-06 | Elapsed: 3.8s
  Step  1000 | Time  2.777 | L2 Error: 1.0824e-05 | Elapsed: 7.7s
  Step  1131 | Time  3.140 | L2 Error: 1.1204e-05 | Elapsed: 8.7s
Simulation Finished! Plotting results...
